# Part III: Quantifying Impact of ESG on Financial Performance

Does doing good mean doing well? 
- Report the regression results of financial performance on lagged ESG scores. 
- Does ESG predict future ROA after controlling for observable firm characteristics? Does ESG predict stock returns either unconditionally or after controlling for earnings news? 
- Please discuss your findings in the framework of financial materiality and impact materiality.

### Set-Up

In [11]:
import wrds
db = wrds.Connection()

Loading library list...
Done


In [1]:
# !pip install --quiet statsmodels seaborn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import statsmodels.formula.api as smf
from IPython.display import display

try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

# Load the panel produced by data_prepare.ipynb.
# This notebook requires the extended CSV that includes dltt, dlc, xrd, and ppent.
# If those columns are missing, re-run data_prepare.ipynb first.
panel = pd.read_csv('esg_financial_panel_2013_2023.csv')
panel['datadate'] = pd.to_datetime(panel['datadate'])
panel['sic'] = pd.to_numeric(panel['sic'], errors='coerce')

required_ext = ['dltt', 'dlc', 'xrd', 'ppent']
missing_ext = [c for c in required_ext if c not in panel.columns]
if missing_ext:
    raise ValueError(
        f"Columns {missing_ext} are missing. "
        "Please re-run data_prepare.ipynb to regenerate "
        "esg_financial_panel_2013_2023.csv with the extended variable set."
    )

print(f'Loaded: {len(panel):,} firm-year observations')
print(f'Columns: {list(panel.columns)}')

Loaded: 20,758 firm-year observations
Columns: ['gvkey', 'permno', 'datadate', 'fyear', 'cusip8', 'sic', 'at', 'sale', 'ni', 'ceq', 'dltt', 'dlc', 'xrd', 'ppent', 'roa', 'size', 'lag_roa', 'lag_size', 'esg_score', 'future_roa', 'future_annual_ret', 'future_n_months_ret']


### Variable Construction

In [2]:
# Fama-French 48 industry classification, consistent with part1.ipynb
# Source: https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/det_48_ind_port.html
FF48 = [
    (1,  'Agriculture',              [(100,199),(200,299),(700,799),(910,919),(2048,2048)]),
    (2,  'Food Products',            [(2000,2046),(2050,2063),(2070,2079),(2086,2086),(2090,2092),(2095,2095),(2096,2096),(2097,2099)]),
    (3,  'Candy & Soda',             [(2064,2068),(2086,2086),(2096,2096),(2097,2097)]),
    (4,  'Beer & Liquor',            [(2080,2085),(2080,2080)]),
    (5,  'Tobacco Products',         [(2100,2199)]),
    (6,  'Recreation',               [(7800,7833),(7840,7841),(7900,7999),(7993,7993),(7997,7997)]),
    (7,  'Entertainment',            [(7812,7819),(7820,7823),(7824,7829)]),
    (8,  'Printing & Publishing',    [(2700,2749),(2770,2771),(2780,2799)]),
    (9,  'Consumer Goods',           [(2047,2047),(2391,2392),(2510,2519),(2590,2599),(2840,2844),(2846,2849),(3160,3162),(3170,3172),(3190,3199),(3229,3229),(3260,3260),(3262,3263),(3269,3269),(3630,3639),(3750,3751),(3800,3800),(3860,3861),(3870,3879),(3910,3919),(3960,3969),(3991,3991),(3995,3995)]),
    (10, 'Apparel',                  [(2300,2390),(3020,3021),(3100,3111),(3130,3131),(3140,3149),(3150,3151),(3963,3965)]),
    (11, 'Healthcare',               [(8000,8099)]),
    (12, 'Medical Equipment',        [(3841,3851),(3841,3841),(3842,3842),(3843,3843),(3844,3844),(3845,3845),(3851,3851),(5047,5047),(5122,5122)]),
    (13, 'Pharmaceutical Products',  [(2830,2836)]),
    (14, 'Chemicals',                [(2800,2829),(2850,2879),(2890,2899)]),
    (15, 'Rubber & Plastic',         [(3000,3030),(3041,3041),(3050,3053),(3060,3069),(3070,3079),(3080,3089),(3090,3099)]),
    (16, 'Textiles',                 [(2200,2284),(2290,2295),(2297,2299),(2393,2395),(2397,2399)]),
    (17, 'Construction Materials',   [(800,899),(1500,1511),(1520,1542),(1550,1559),(1600,1699),(1711,1799),(2400,2439),(2450,2459),(2490,2499),(2660,2661),(2950,2952),(3200,3200),(3210,3211),(3240,3241),(3250,3259),(3261,3261),(3264,3264),(3270,3275),(3280,3281),(3290,3293),(3295,3299),(3420,3429),(3430,3433),(3440,3441),(3442,3442),(3446,3446),(3448,3448),(3449,3449),(3460,3469),(3490,3499),(3996,3996)]),
    (18, 'Steel Works',              [(3300,3300),(3310,3317),(3320,3325),(3330,3341),(3350,3357),(3360,3369),(3390,3399)]),
    (19, 'Fabricated Products',      [(3410,3412),(3443,3443),(3444,3444),(3460,3462),(3490,3492),(3494,3495),(3496,3499)]),
    (20, 'Machinery',                [(3510,3519),(3520,3529),(3530,3539),(3540,3549),(3550,3559),(3560,3569),(3580,3589),(3590,3599)]),
    (21, 'Electrical Equipment',     [(3600,3600),(3610,3613),(3620,3621),(3623,3629),(3640,3646),(3648,3649),(3660,3660),(3690,3692),(3699,3699)]),
    (22, 'Automobiles & Trucks',     [(2296,2296),(2396,2396),(3010,3011),(3537,3537),(3647,3647),(3694,3694),(3700,3716),(3750,3751),(3790,3792),(3799,3799)]),
    (23, 'Aircraft',                 [(3720,3721),(3723,3725),(3728,3729)]),
    (24, 'Shipbuilding',             [(3730,3731),(3740,3743)]),
    (25, 'Defense',                  [(3760,3769),(3480,3489),(3812,3812)]),
    (26, 'Precious Metals',          [(1040,1049)]),
    (27, 'Non-Metallic Mining',      [(1000,1039),(1050,1099),(1400,1499)]),
    (28, 'Coal',                     [(1200,1299)]),
    (29, 'Petroleum & Natural Gas',  [(1300,1399),(2900,2912),(2990,2999)]),
    (30, 'Utilities',                [(4900,4942),(4950,4952),(4953,4953),(4959,4959),(4961,4961),(4991,4991)]),
    (31, 'Communication',            [(4800,4899)]),
    (32, 'Personal Services',        [(7020,7021),(7040,7041),(7080,7081),(7200,7299),(7300,7300),(7389,7389),(7395,7395),(7500,7521),(7532,7534),(7536,7599),(7600,7641),(7690,7699),(8200,8499),(8600,8699),(8800,8899),(7510,7515)]),
    (33, 'Business Services',        [(7370,7379),(7380,7399),(7514,7515),(7519,7519),(8700,8748),(8900,8999),(4220,4229)]),
    (34, 'Computers',                [(3570,3579),(3680,3689),(3695,3695),(7373,7373)]),
    (35, 'Electronic Equipment',     [(3622,3622),(3661,3669),(3670,3679),(3810,3810),(3812,3812)]),
    (36, 'Measuring Instruments',    [(3811,3811),(3820,3829),(3830,3839)]),
    (37, 'Misc. Business',           [(3900,3999)]),
    (38, 'Transportation',           [(4000,4013),(4040,4049),(4100,4231),(4400,4499),(4600,4621),(4700,4789)]),
    (39, 'Wholesale',                [(5000,5199)]),
    (40, 'Retail',                   [(5200,5999)]),
    (41, 'Restaurants & Hotels',     [(5800,5829),(5890,5899),(7000,7011),(7041,7041)]),
    (42, 'Banking',                  [(6000,6199)]),
    (43, 'Insurance',                [(6300,6411)]),
    (44, 'Real Estate',              [(6500,6515),(6552,6553),(6798,6798)]),
    (45, 'Trading',                  [(6200,6299),(6710,6799)]),
]

sic_to_ff48 = {}
for ff_id, ff_name, ranges in FF48:
    for lo, hi in ranges:
        for s in range(lo, hi + 1):
            if s not in sic_to_ff48:
                sic_to_ff48[s] = (ff_id, ff_name)

def assign_ff48(sic):
    if pd.isna(sic):
        return (48, 'Other')
    return sic_to_ff48.get(int(sic), (48, 'Other'))

ff48_result = panel['sic'].apply(assign_ff48)
panel['ff48_id']  = ff48_result.apply(lambda x: x[0]).astype(int)
panel['industry'] = ff48_result.apply(lambda x: x[1])

print(f"Industry assignment complete. Unclassified: {(panel['industry'] == 'Other').sum():,} "
      f"({(panel['industry'] == 'Other').mean() * 100:.1f}%)")

Industry assignment complete. Unclassified: 1,009 (4.9%)


In [3]:
# Sort panel for computing lags within each firm
panel = panel.sort_values(['gvkey', 'fyear']).copy()

# Leverage: total debt (long-term + current) scaled by total assets.
# Missing debt fields are set to zero on the assumption that non-reporting
# reflects the absence of recorded debt obligations (consistent with, e.g.,
# Rajan and Zingales, 1995).
panel['leverage'] = (
    panel['dltt'].fillna(0) + panel['dlc'].fillna(0)
) / panel['at']

# R&D intensity: R&D expense scaled by total assets.
# Firms that do not report xrd are assigned zero, following the convention
# established in the innovation and intangibles literature.
panel['rd_intensity'] = panel['xrd'].fillna(0) / panel['at']

# Capital intensity: net PP&E scaled by total assets.
# Captures the degree to which a firm's asset base consists of tangible
# physical capital subject to environmental regulations.
panel['capital_intensity'] = panel['ppent'].fillna(0) / panel['at']

# Equity ratio: common equity scaled by total assets.
# Reflects balance sheet health; negative values occur for firms with
# accumulated losses exceeding paid-in capital.
panel['equity_ratio'] = panel['ceq'] / panel['at']

# Profit margin: net income scaled by net sales.
# Restricted to firm-years with positive sales to avoid undefined or
# economically uninterpretable values.
panel['profit_margin'] = np.where(
    panel['sale'] > 0,
    panel['ni'] / panel['sale'],
    np.nan
)

# Sales growth: year-over-year percentage change in net sales.
# Lagged sales are computed within each firm. Firm-years where
# lagged sales are non-positive are excluded.
panel['lag_sale'] = panel.groupby('gvkey')['sale'].shift(1)
panel['sales_growth'] = np.where(
    panel['lag_sale'] > 0,
    (panel['sale'] - panel['lag_sale']) / panel['lag_sale'],
    np.nan
)

# Report coverage for derived variables
new_vars = ['leverage', 'rd_intensity', 'capital_intensity',
            'equity_ratio', 'profit_margin', 'sales_growth']
print('Non-missing observations for derived variables:')
for v in new_vars:
    print(f'  {v:<22}: {panel[v].notna().sum():>6,}')

Non-missing observations for derived variables:
  leverage              : 20,758
  rd_intensity          : 20,758
  capital_intensity     : 20,758
  equity_ratio          : 20,758
  profit_margin         : 20,199
  sales_growth          : 16,996


In [4]:
panel = panel.sort_values(['gvkey', 'fyear']).copy()

def winsorize(series, lower=0.01, upper=0.99):
    """Winsorize a pandas Series at the specified quantile bounds."""
    lo = series.quantile(lower)
    hi = series.quantile(upper)
    return series.clip(lo, hi)

# roa and size are raw in the CSV (winsorization in part1.ipynb was in-memory only).
# All derived variables are winsorized here before use in any table or regression.
winsorize_vars = [
    'roa', 'size',
    'leverage', 'rd_intensity', 'capital_intensity',
    'equity_ratio', 'profit_margin', 'sales_growth',
    'future_roa', 'future_annual_ret',          # outcome variables, first used in Part 3
]
for v in winsorize_vars:
    panel[v] = winsorize(panel[v])

print('Winsorization complete (1st and 99th percentiles).')

# earnings_news constructed after winsorization so it inherits the cleaned roa values
panel['earnings_news'] = panel['future_roa'] - panel['roa']

panel['fyear_str']   = panel['fyear'].astype(str)
panel['ff48_id_str'] = panel['ff48_id'].astype(str)

Winsorization complete (1st and 99th percentiles).


In [5]:
for col in panel.columns:
    print(f'{col:<20}  {panel[col].dtype}')

gvkey                 int64
permno                float64
datadate              datetime64[ns]
fyear                 int64
cusip8                object
sic                   float64
at                    float64
sale                  float64
ni                    float64
ceq                   float64
dltt                  float64
dlc                   float64
xrd                   float64
ppent                 float64
roa                   float64
size                  float64
lag_roa               float64
lag_size              float64
esg_score             float64
future_roa            float64
future_annual_ret     float64
future_n_months_ret   int64
ff48_id               int64
industry              object
leverage              float64
rd_intensity          float64
capital_intensity     float64
equity_ratio          float64
profit_margin         float64
lag_sale              float64
sales_growth          float64
earnings_news         float64
fyear_str             object
ff48_id_str   

In [6]:
panel

,gvkey,permno,datadate,fyear,cusip8,sic,at,sale,ni,ceq,...,leverage,rd_intensity,capital_intensity,equity_ratio,profit_margin,lag_sale,sales_growth,earnings_news,fyear_str,ff48_id_str
0,1004,54594.0,2017-05-31,2016,00036110,5080.0,1504.100,1767.600,56.500,914.200,...,0.104581,0.000000,0.240343,0.607805,0.031964,NaN,NaN,-0.027332,2016,39
1,1004,54594.0,2018-05-31,2017,00036110,5080.0,1524.700,1748.300,15.600,936.300,...,0.116220,0.000000,0.207647,0.614088,0.008923,1767.600,-0.010919,-0.005288,2017,39
2,1004,54594.0,2019-05-31,2018,00036110,5080.0,1517.200,2051.800,7.500,905.900,...,0.093396,0.000000,0.229897,0.597087,0.003655,1748.300,0.173597,-0.002827,2018,39
3,1004,54594.0,2020-05-31,2019,00036110,5080.0,2079.000,2089.300,4.400,902.600,...,0.329293,0.000000,0.210245,0.434151,0.002106,2051.800,0.018277,0.021135,2019,39
4,1004,54594.0,2021-05-31,2020,00036110,5080.0,1539.700,1651.400,35.800,974.400,...,0.133208,0.000000,0.246866,0.632851,0.021679,2089.300,-0.209592,0.026752,2020,39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20753,339965,19654.0,2022-01-31,2021,83344510,7370.0,6649.698,1219.327,-679.948,5049.045,...,0.031024,0.070219,0.044428,0.759289,-0.557642,NaN,NaN,-0.000917,2021,33
20754,339965,19654.0,2023-01-31,2022,83344510,7370.0,7722.322,2065.659,-796.705,5456.436,...,0.032588,0.102049,0.050773,0.706580,-0.385690,1219.327,0.694098,0.001496,2022,33
20755,345556,16069.0,2021-12-31,2021,30315R10,2836.0,123.021,21.167,-31.283,96.857,...,0.107575,0.233700,0.033880,0.787321,-1.477914,NaN,NaN,NaN,2021,13
20756,347007,15533.0,2022-12-31,2022,45256X10,2836.0,362.356,0.240,-416.567,-447.327,...,1.019711,0.553478,0.523193,-0.453973,-34.603915,NaN,NaN,0.018635,2022,13


In [8]:
panel['esg_score'].describe()

count    20758.000000
mean         0.404152
std          0.190924
min          0.069560
25%          0.253776
50%          0.373490
75%          0.538747
max          0.854176
Name: esg_score, dtype: float64

### Regression 1: Financial Performance on Lagged ESG Scores

In [9]:
# Create copy of panel with only the variables needed for regression tables to speed up estimation
reg1_vars = ['future_roa', 'esg_score', 'fyear_str', 'ff48_id_str']
sample_reg1 = panel.dropna(subset=reg1_vars).copy()

# Lag regression of financial performance on lagged ESG scores
reg1 = smf.ols(
    'future_roa ~ esg_score + C(fyear_str) + C(ff48_id_str)',
    data=sample_reg1
).fit(cov_type='HC3')

print("=== Regression 1: Future ROA ~ ESG Score (No Firm-Level Controls) ===")
print(f"N = {int(reg1.nobs):,}   R² = {reg1.rsquared:.4f}")
print(f"\n{'Variable':<20} {'Coef':>10} {'Std Err':>10} {'t':>8} {'p-value':>10}")
print("-" * 62)
for var in ['esg_score']:
    coef  = reg1.params[var]
    se    = reg1.bse[var]
    t     = reg1.tvalues[var]
    p     = reg1.pvalues[var]
    stars = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''
    print(f"{var:<20} {coef:>10.4f} {se:>10.4f} {t:>8.2f} {p:>10.4f} {stars}")
print("\nYear FE: Yes    Industry FE: Yes    Controls: No")

=== Regression 1: Future ROA ~ ESG Score (No Firm-Level Controls) ===
N = 18,372   R² = 0.3323

Variable                   Coef    Std Err        t    p-value
--------------------------------------------------------------
esg_score                0.1526     0.0057    26.89     0.0000 ***

Year FE: Yes    Industry FE: Yes    Controls: No


### Regression 2: Future ROA on ESG Scores (Controlling for Firm Characteristics)

In [10]:
# Create copy of panel with all variables needed for regression tables to speed up estimation
reg2_vars = ['future_roa', 'esg_score', 'size', 'roa', 
             'leverage', 'rd_intensity', 'capital_intensity',
             'sales_growth', 'fyear_str', 'ff48_id_str']
sample_reg2 = panel.dropna(subset=reg2_vars).copy()

# Lag regression of financial performance on lagged ESG scores with firm-level controls
reg2 = smf.ols(
    'future_roa ~ esg_score + size + roa + leverage '
    '+ rd_intensity + capital_intensity + sales_growth + C(fyear_str) + C(ff48_id_str)',
    data=sample_reg2
).fit(cov_type='HC3')

print("=== Regression 2: Future ROA ~ ESG Score + Firm Controls ===")
print(f"N = {int(reg2.nobs):,}   R² = {reg2.rsquared:.4f}")
print(f"\n{'Variable':<22} {'Coef':>10} {'Std Err':>10} {'t':>8} {'p-value':>10}")
print("-" * 64)
for var in ['esg_score', 'size', 'roa', 'leverage',
            'rd_intensity', 'capital_intensity', 'sales_growth']:
    coef  = reg2.params[var]
    se    = reg2.bse[var]
    t     = reg2.tvalues[var]
    p     = reg2.pvalues[var]
    stars = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''
    print(f"{var:<22} {coef:>10.4f} {se:>10.4f} {t:>8.2f} {p:>10.4f} {stars}")
print("\nYear FE: Yes    Industry FE: Yes    Controls: Yes")

=== Regression 2: Future ROA ~ ESG Score + Firm Controls ===
N = 14,834   R² = 0.5753

Variable                     Coef    Std Err        t    p-value
----------------------------------------------------------------
esg_score                  0.0311     0.0050     6.16     0.0000 ***
size                       0.0007     0.0006     1.13     0.2588 
roa                        0.6613     0.0157    42.01     0.0000 ***
leverage                   0.0210     0.0053     3.94     0.0001 ***
rd_intensity              -0.1021     0.0314    -3.25     0.0012 ***
capital_intensity          0.0047     0.0052     0.90     0.3680 
sales_growth              -0.0142     0.0047    -3.02     0.0026 ***

Year FE: Yes    Industry FE: Yes    Controls: Yes


### Regression 3: Stock Returns on ESG Scores (Unconditionally or After Controlling For Earnings News)

##### Proxy Earnings News

In [14]:
# 3a: Unconditional
ret1_vars = ['future_annual_ret', 'esg_score', 'fyear_str', 'ff48_id_str']
sample_ret1 = panel.dropna(subset=ret1_vars).copy()

reg_ret1 = smf.ols(
    'future_annual_ret ~ esg_score + C(fyear_str) + C(ff48_id_str)',
    data=sample_ret1
).fit(cov_type='HC3')

# 3b: Controlling for earnings news
ret2_vars = ['future_annual_ret', 'esg_score', 'earnings_news',
             'roa', 'size', 'leverage', 'sales_growth',
             'rd_intensity', 'capital_intensity',
             'fyear_str', 'ff48_id_str']
sample_ret2 = panel.dropna(subset=ret2_vars).copy()

reg_ret2 = smf.ols(
    'future_annual_ret ~ esg_score + earnings_news + roa '
    '+ size + leverage + sales_growth '
    '+ rd_intensity + capital_intensity '
    '+ C(fyear_str) + C(ff48_id_str)',
    data=sample_ret2
).fit(cov_type='HC3')

print("=== Regression 3a: Future Return ~ ESG Score (Unconditional) ===")
print(f"N = {int(reg_ret1.nobs):,}   R² = {reg_ret1.rsquared:.4f}")
coef  = reg_ret1.params['esg_score']
se    = reg_ret1.bse['esg_score']
p     = reg_ret1.pvalues['esg_score']
stars = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''
print(f"  esg_score: {coef:.4f} ({se:.4f})  p={p:.4f} {stars}")

print("\n=== Regression 3b: Future Return ~ ESG + Earnings News + Controls ===")
print(f"N = {int(reg_ret2.nobs):,}   R² = {reg_ret2.rsquared:.4f}")
print(f"\n{'Variable':<22} {'Coef':>10} {'Std Err':>10} {'t':>8} {'p-value':>10}")
print("-" * 64)
for var in ['esg_score', 'earnings_news', 'roa', 'size', 'leverage', 'sales_growth', 'rd_intensity', 'capital_intensity']:
    coef  = reg_ret2.params[var]
    se    = reg_ret2.bse[var]
    t     = reg_ret2.tvalues[var]
    p     = reg_ret2.pvalues[var]
    stars = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''
    print(f"{var:<22} {coef:>10.4f} {se:>10.4f} {t:>8.2f} {p:>10.4f} {stars}")
print("\nYear FE: Yes    Industry FE: Yes    Controls: Yes")

=== Regression 3a: Future Return ~ ESG Score (Unconditional) ===
N = 20,758   R² = 0.0778
  esg_score: -0.0360 (0.0173)  p=0.0380 **

=== Regression 3b: Future Return ~ ESG + Earnings News + Controls ===
N = 14,834   R² = 0.1884

Variable                     Coef    Std Err        t    p-value
----------------------------------------------------------------
esg_score                 -0.0240     0.0246    -0.97     0.3305 
earnings_news              1.4029     0.0639    21.97     0.0000 ***
roa                        0.4230     0.0462     9.15     0.0000 ***
size                      -0.0035     0.0029    -1.18     0.2371 
leverage                   0.0479     0.0220     2.17     0.0299 **
sales_growth              -0.0273     0.0139    -1.97     0.0491 **
rd_intensity               0.5768     0.1046     5.51     0.0000 ***
capital_intensity          0.0071     0.0268     0.27     0.7907 

Year FE: Yes    Industry FE: Yes    Controls: Yes


### Discussion of findings in the framework of financial materiality and impact materiality.
